<a href="https://colab.research.google.com/github/amrithavi/GENE-PAPER/blob/main/DNALSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q tensorflow pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
from google.colab import files
files.upload()

MessageError: CustomError: Timed out waiting for iframe configuration. URL: https://colab.research.google.com/drive/1mMaA3-0a3ha11K5OAPq3iRJBiiEJKYof#scrollTo=2uo8J_36H-tT

In [ ]:
import zipfile, os

zip_path = "/content/archive (3).zip"   # <-- IMPORTANT

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/data")

print(os.listdir("/content/data"))

In [ ]:
train_df = pd.read_csv("/content/data/train.csv")
val_df   = pd.read_csv("/content/data/validation.csv")
test_df  = pd.read_csv("/content/data/test.csv")

train_df.head()

In [ ]:
def clean_df(df):
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    df = df.dropna()
    return df

train_df = clean_df(train_df)
val_df   = clean_df(val_df)
test_df  = clean_df(test_df)

In [ ]:
dna_tokenizer = Tokenizer(char_level=True)
dna_tokenizer.fit_on_texts(train_df["NucleotideSequence"])

X_train_seq = dna_tokenizer.texts_to_sequences(train_df["NucleotideSequence"])
X_val_seq   = dna_tokenizer.texts_to_sequences(val_df["NucleotideSequence"])
X_test_seq  = dna_tokenizer.texts_to_sequences(test_df["NucleotideSequence"])

MAX_LEN = 500

X_train_seq = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq   = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq  = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)

X_train_text = tfidf.fit_transform(train_df["Description"]).toarray()
X_val_text   = tfidf.transform(val_df["Description"]).toarray()
X_test_text  = tfidf.transform(test_df["Description"]).toarray()

In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(train_df["GeneType"])
y_val   = le.transform(val_df["GeneType"])
y_test  = le.transform(test_df["GeneType"])

num_classes = len(le.classes_)

In [ ]:
# --- DNA Branch ---
seq_input = Input(shape=(MAX_LEN,))
x = Embedding(input_dim=len(dna_tokenizer.word_index)+1, output_dim=8)(seq_input)
x = LSTM(64)(x)
x = Dropout(0.3)(x)

# --- Text Branch ---
text_input = Input(shape=(X_train_text.shape[1],))
y = Dense(256, activation="relu")(text_input)
y = Dropout(0.3)(y)

# --- Fusion ---
combined = Concatenate()([x, y])
z = Dense(128, activation="relu")(combined)
z = Dropout(0.3)(z)
output = Dense(num_classes, activation="softmax")(z)

model = Model(inputs=[seq_input, text_input], outputs=output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    [X_train_seq, X_train_text],
    y_train,
    validation_data=([X_val_seq, X_val_text], y_val),
    epochs=10,
    batch_size=64
)

In [ ]:
pred = model.predict([X_test_seq, X_test_text])
pred_classes = np.argmax(pred, axis=1)

print("Accuracy:", accuracy_score(y_test, pred_classes))
print(classification_report(y_test, pred_classes, target_names=le.classes_))

In [ ]:
def predict_gene(sequence, description):
    seq = dna_tokenizer.texts_to_sequences([sequence])
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

    text = tfidf.transform([description]).toarray()

    pred = model.predict([seq, text])
    return le.inverse_transform([np.argmax(pred)])

predict_gene("ATCGGCTAGCTAGCTA", "regulatory region involved in transcription")

In [ ]:
import matplotlib.pyplot as plt

# Plot Accuracy
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.xlabel("Epoch Count")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend(["Training Accuracy", "Validation Accuracy"])

plt.show()